# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds a transparent Week-4 baseline from observable, pre-decision signals only. The rule is deliberately simple and frozen before Week-5 modeling.

**Lane:** content refresh / search visibility

**Decision slice:** bundled anonymized starter dataset. The starter slice is a trailing-90-day snapshot; it contains no FlyRank product flags, so this audit validates observable proxies rather than reverse-engineering a hidden product score.

## 1. Signal checks and rule reasoning

### Signal 1 — staleness
**Hypothesis:** pages that have not been updated for a long time are more plausible refresh candidates. This is the observable signal behind the session's refresh/staleness flag logic. I will bucket `days_since_last_update` and compare each bucket's declining rate, using `n` in every bucket. A positive step-up across older buckets supports the hypothesis; a flat or reversed pattern is evidence against treating staleness as strong by itself.

**Verdict:** CONFIRMED / OPPOSITE / MIXED / FALSE (filled by the executed cell below).

### Signal 2 — search visibility / volume
**Hypothesis:** a refresh is more actionable when the page has meaningful search exposure. This is the observable signal behind the session's quick-win / volume logic. I will bucket `impressions_90d` and compare the same outcome rate, again printing `n`. Because low-volume position and CTR are noisy, volume is used as an opportunity gate rather than as proof of decline.

**Verdict:** CONFIRMED / OPPOSITE / MIXED / FALSE (filled by the executed cell below).

### Rule in plain words
A page rises to the top of the baseline queue when it is **old since its last update** and has **enough search visibility to make a refresh worth reviewing**. Among pages meeting those conditions, more visible pages rank first. The rule does not use `trend_pct`, `trend_direction`, `is_declining_label`, or any future-window field.

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# 1. LOAD DATA
# ============================================================

ROOT = Path.cwd()

# Find repository root automatically
candidates = [
    ROOT,
    ROOT / "ML_Pipeline",
    ROOT.parent,
    ROOT.parent.parent,
]

for candidate in candidates:
    if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
        ROOT = candidate
        break

CSV_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
OUT_DIR = ROOT / "work/outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)
print("Dataset:", CSV_PATH)

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "word_count",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

numeric_cols = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "word_count",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# ============================================================
# 3. AUDIT OUTCOME
# ============================================================
# Used ONLY to check whether signals are useful.
# It is NOT used in the baseline score.

if "is_declining_label" in df.columns:
    audit_y = pd.to_numeric(
        df["is_declining_label"],
        errors="coerce"
    ).fillna(0).astype(int)

elif "trend_direction" in df.columns:
    audit_y = (
        df["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("down")
        .astype(int)
    )

else:
    raise ValueError(
        "Need is_declining_label or trend_direction for the audit."
    )

print(
    f"Overall declining rate for audit: {audit_y.mean():.3f}"
)


# ============================================================
# 4. SIGNAL 1 — STALENESS
# ============================================================

print("\n" + "=" * 60)
print("SIGNAL 1 — STALENESS")
print("=" * 60)

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"],
)

stale_tab = (
    pd.DataFrame({
        "bucket": df["staleness_bucket"],
        "audit_outcome": audit_y
    })
    .dropna(subset=["bucket"])
    .groupby("bucket", observed=False)["audit_outcome"]
    .agg(
        n="size",
        declining_rate="mean"
    )
    .reset_index()
)

print("\nStaleness bucket table:")
display(stale_tab)

stale_rates = stale_tab["declining_rate"].to_numpy()

if len(stale_rates) >= 3:
    diffs = np.diff(stale_rates)

    if (
        np.all(diffs >= -0.01)
        and stale_rates[-1] - stale_rates[0] >= 0.05
    ):
        stale_verdict = "CONFIRMED"

    elif (
        np.all(diffs <= 0.01)
        and stale_rates[0] - stale_rates[-1] >= 0.05
    ):
        stale_verdict = "OPPOSITE"

    else:
        stale_verdict = "MIXED"
else:
    stale_verdict = "FALSE"

print(f"VERDICT — STALENESS: {stale_verdict}")


# ============================================================
# 5. SIGNAL 2 — SEARCH VISIBILITY / VOLUME
# ============================================================

print("\n" + "=" * 60)
print("SIGNAL 2 — SEARCH VISIBILITY / VOLUME")
print("=" * 60)

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 299, 2999, 29999, np.inf],
    labels=[
        "<100",
        "100-299",
        "300-2999",
        "3000-29999",
        "30000+"
    ],
)

volume_tab = (
    pd.DataFrame({
        "bucket": df["volume_bucket"],
        "audit_outcome": audit_y
    })
    .dropna(subset=["bucket"])
    .groupby("bucket", observed=False)["audit_outcome"]
    .agg(
        n="size",
        declining_rate="mean"
    )
    .reset_index()
)

print("\nVolume bucket table:")
display(volume_tab)

volume_rates = volume_tab["declining_rate"].to_numpy()

if len(volume_rates) >= 3:
    diffs = np.diff(volume_rates)

    if (
        np.all(diffs >= -0.01)
        and volume_rates[-1] - volume_rates[0] >= 0.05
    ):
        volume_verdict = "CONFIRMED"

    elif (
        np.all(diffs <= 0.01)
        and volume_rates[0] - volume_rates[-1] >= 0.05
    ):
        volume_verdict = "OPPOSITE"

    else:
        volume_verdict = "MIXED"
else:
    volume_verdict = "FALSE"

print(f"VERDICT — VOLUME: {volume_verdict}")


# ============================================================
# 6. BASELINE RULE
# ============================================================

print("\n" + "=" * 60)
print("BASELINE RULE")
print("=" * 60)

print("""
Rule:
    stale    = days_since_last_update >= 180
    visible  = impressions_90d >= 500
    score    = stale * visible * log1p(impressions_90d)

Action:
    stale + visible -> refresh
    otherwise       -> monitor

Reason codes:
    stale_visible_page
    general_review
""")

work = df.copy()

work["stale"] = (
    work["days_since_last_update"] >= 180
).astype(int)

work["visible"] = (
    work["impressions_90d"] >= 500
).astype(int)

work["score"] = (
    work["stale"]
    * work["visible"]
    * np.log1p(
        work["impressions_90d"].clip(lower=0)
    )
)

work["reason_code"] = np.where(
    (
        work["stale"].eq(1)
        & work["visible"].eq(1)
    ),
    "stale_visible_page",
    "general_review"
)

work["action"] = np.where(
    work["reason_code"].eq("stale_visible_page"),
    "refresh",
    "monitor"
)


# ============================================================
# 7. RANK
# ============================================================

work = work.sort_values(
    [
        "score",
        "impressions_90d",
        "days_since_last_update"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)


# ============================================================
# 8. DIAGNOSTICS
# ============================================================

work["audit_label"] = audit_y.reset_index(drop=True)

def precision_at_k(frame, k):
    if len(frame) == 0:
        return 0.0

    return float(
        frame.head(k)["audit_label"].mean()
    )

base_rate = float(
    work["audit_label"].mean()
)

p10 = precision_at_k(work, 10)
p50 = precision_at_k(work, 50)

print("\n" + "=" * 60)
print("BASELINE DIAGNOSTICS")
print("=" * 60)

print(f"Base rate    : {base_rate:.3f}")
print(f"Precision@10 : {p10:.3f}")
print(f"Precision@50 : {p50:.3f}")


# ============================================================
# 9. WRITE REQUIRED CSV
# ============================================================

queue_columns = [
    "rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "content_age_days",
    "word_count",
]

queue = work[queue_columns].copy()

output_path = (
    OUT_DIR / "baseline_action_score.csv"
)

queue.to_csv(
    output_path,
    index=False
)

print("\n" + "=" * 60)
print("QUEUE CREATED")
print("=" * 60)

print(f"Wrote {len(queue):,} rows")
print(f"Path: {output_path}")


# ============================================================
# 10. RECEIPT JSON
# ============================================================

receipt = {
    "rows": int(len(queue)),
    "base_rate_declining": base_rate,
    "precision_at_10": p10,
    "precision_at_50": p50,
    "staleness_verdict": stale_verdict,
    "volume_verdict": volume_verdict,
    "score_inputs": [
        "days_since_last_update",
        "impressions_90d"
    ],
    "reason_codes": [
        "stale_visible_page",
        "general_review"
    ],
    "actions": [
        "refresh",
        "monitor"
    ],
    "leakage_note": (
        "The baseline score does not use "
        "trend_pct, trend_direction, "
        "is_declining_label, or future-window fields."
    )
}

receipt_path = (
    OUT_DIR / "w04_baseline_receipt.json"
)

receipt_path.write_text(
    json.dumps(receipt, indent=2)
)

print(f"Receipt: {receipt_path}")


# ============================================================
# 11. TOP-10 SKEPTIC REVIEW
# ============================================================

print("\n" + "=" * 60)
print("TOP-10 SKEPTIC REVIEW")
print("=" * 60)

review_rows = []

for _, row in work.head(10).iterrows():

    if row["reason_code"] == "stale_visible_page":

        action = "refresh"

        why = (
            f"old update "
            f"({int(row['days_since_last_update'])}d) "
            f"+ visible "
            f"({int(row['impressions_90d']):,} impressions)"
        )

        wrong = (
            "Could be wrong if the page is intentionally "
            "evergreen, already scheduled for refresh, "
            "or the traffic is low-quality."
        )

    else:

        action = "monitor"

        why = (
            "does not meet both baseline gates"
        )

        wrong = (
            "Could be wrong if manual review finds "
            "a business priority not captured by these signals."
        )

    review_rows.append({
        "rank": int(row["rank"]),
        "action": action,
        "why_it_is_here": why,
        "what_would_make_it_wrong": wrong
    })

review = pd.DataFrame(review_rows)

display(review)


# ============================================================
# 12. LEAKAGE CHECK
# ============================================================

print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

score_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

print("Score inputs:")
for item in score_inputs:
    print(f"  ✓ {item}")

print("\nNot used in score:")
print("  ✓ trend_pct")
print("  ✓ trend_direction")
print("  ✓ is_declining_label")
print("  ✓ future-window fields")

print("\nLeakage check: PASSED")


# ============================================================
# 13. FINAL CHECKS
# ============================================================

assert len(queue) == len(df)

assert output_path.exists()

assert set(queue["reason_code"].unique()).issubset(
    {
        "stale_visible_page",
        "general_review"
    }
)

assert set(queue["action"].unique()).issubset(
    {
        "refresh",
        "monitor"
    }
)

assert queue["rank"].is_monotonic_increasing

print("\n" + "=" * 60)
print("FINAL CHECKS")
print("=" * 60)

print("✓ Row count preserved")
print("✓ Reason codes valid")
print("✓ Action labels valid")
print("✓ Queue ranked")
print("✓ CSV written")
print("✓ Receipt written")
print("✓ Leakage check passed")

print("\nDONE — Week-4 baseline completed.")

Repository: /Users/hardikk/Desktop/ML_Pipeline/ML_Pipeline
Dataset: /Users/hardikk/Desktop/ML_Pipeline/ML_Pipeline/data/raw/content_refresh_anonymized.csv
Loaded 30,000 rows and 44 columns
Overall declining rate for audit: 0.542

SIGNAL 1 — STALENESS

Staleness bucket table:


,bucket,n,declining_rate
0,0-30,20480,0.511377
1,31-90,175,0.588571
2,91-180,9171,0.611057
3,181-365,169,0.467456
4,365+,5,0.600000


VERDICT — STALENESS: MIXED

SIGNAL 2 — SEARCH VISIBILITY / VOLUME

Volume bucket table:


,bucket,n,declining_rate
0,<100,7994,0.389042
1,100-299,3254,0.613399
2,300-2999,10469,0.614672
3,3000-29999,7205,0.586121
4,30000+,1078,0.461967


VERDICT — VOLUME: MIXED

BASELINE RULE

Rule:
    stale    = days_since_last_update >= 180
    visible  = impressions_90d >= 500
    score    = stale * visible * log1p(impressions_90d)

Action:
    stale + visible -> refresh
    otherwise       -> monitor

Reason codes:
    stale_visible_page
    general_review


BASELINE DIAGNOSTICS
Base rate    : 0.542
Precision@10 : 0.800
Precision@50 : 0.680

QUEUE CREATED
Wrote 30,000 rows
Path: /Users/hardikk/Desktop/ML_Pipeline/ML_Pipeline/work/outputs/baseline_action_score.csv
Receipt: /Users/hardikk/Desktop/ML_Pipeline/ML_Pipeline/work/outputs/w04_baseline_receipt.json

TOP-10 SKEPTIC REVIEW


,rank,action,why_it_is_here,what_would_make_it_wrong
0,1,refresh,"old update (194d) + visible (61,678 impressions)",Could be wrong if the page is intentionally ev...
1,2,refresh,"old update (194d) + visible (59,472 impressions)",Could be wrong if the page is intentionally ev...
2,3,refresh,"old update (194d) + visible (25,715 impressions)",Could be wrong if the page is intentionally ev...
3,4,refresh,"old update (193d) + visible (13,299 impressions)",Could be wrong if the page is intentionally ev...
4,5,refresh,"old update (194d) + visible (7,812 impressions)",Could be wrong if the page is intentionally ev...
5,6,refresh,"old update (193d) + visible (7,558 impressions)",Could be wrong if the page is intentionally ev...
6,7,refresh,"old update (194d) + visible (4,590 impressions)",Could be wrong if the page is intentionally ev...
7,8,refresh,"old update (194d) + visible (4,556 impressions)",Could be wrong if the page is intentionally ev...
8,9,refresh,"old update (194d) + visible (4,429 impressions)",Could be wrong if the page is intentionally ev...
9,10,refresh,"old update (193d) + visible (1,697 impressions)",Could be wrong if the page is intentionally ev...



LEAKAGE CHECK
Score inputs:
  ✓ days_since_last_update
  ✓ impressions_90d

Not used in score:
  ✓ trend_pct
  ✓ trend_direction
  ✓ is_declining_label
  ✓ future-window fields

Leakage check: PASSED

FINAL CHECKS
✓ Row count preserved
✓ Reason codes valid
✓ Action labels valid
✓ Queue ranked
✓ CSV written
✓ Receipt written
✓ Leakage check passed

DONE — Week-4 baseline completed.


### Rule design choice

The scoring rule below is intentionally not fitted. It uses two simple gates: **stale** = `days_since_last_update >= 180`, and **visible** = `impressions_90d >= 500`. The score is `stale * visible * log1p(impressions_90d)`. This preserves the session's idea: a rule should be human-readable, threshold-based, and easy to challenge.

The rule emits exactly **one reason code**: `stale_visible_page` when both gates fire, otherwise `general_review`. The action label is `refresh` for the flagged group and `monitor` otherwise.

## 2. Build the ranked queue (writes the CSV)

The queue is written to `work/outputs/baseline_action_score.csv`. The CSV is intentionally ignored by git because the repo's leak-guard says to regenerate it from the notebook.

In [3]:
# Transparent baseline: no fitted weights, no trend/label features, no future window.
work = df.copy()
work['stale'] = (work['days_since_last_update'] >= 180).astype(int)
work['visible'] = (work['impressions_90d'] >= 500).astype(int)
work['score'] = work['stale'] * work['visible'] * np.log1p(work['impressions_90d'].clip(lower=0))
work['reason_code'] = np.where(work['stale'].eq(1) & work['visible'].eq(1), 'stale_visible_page', 'general_review')
work['action'] = np.where(work['reason_code'].eq('stale_visible_page'), 'refresh', 'monitor')
work = work.sort_values(['score','impressions_90d','days_since_last_update'], ascending=[False,False,False]).reset_index(drop=True)
work['rank'] = np.arange(1, len(work) + 1)

# Precision@K is diagnostic only; it uses the retrospective label and never enters score creation.
work['audit_label'] = audit_y.reset_index(drop=True)

def precision_at_k(frame: pd.DataFrame, k: int) -> float:
    return float(frame.head(min(k, len(frame)))['audit_label'].mean()) if len(frame) else 0.0

p10 = precision_at_k(work, 10)
p50 = precision_at_k(work, 50)
base_rate = float(work['audit_label'].mean())
print(f'Base rate: {base_rate:.3f}')
print(f'Precision@10: {p10:.3f}')
print(f'Precision@50: {p50:.3f}')

queue_cols = ['rank','content_id','client_id','score','reason_code','action','impressions_90d','days_since_last_update','avg_position','ctr','content_age_days','word_count']
queue = work[queue_cols].copy()
queue.to_csv(OUT_DIR / 'baseline_action_score.csv', index=False)

# Save a small receipt JSON; CSV remains intentionally ignored by git.
receipt = {
    'rows': int(len(queue)),
    'base_rate_declining': base_rate,
    'precision_at_10': p10,
    'precision_at_50': p50,
    'rule': 'stale=(days_since_last_update>=180); visible=(impressions_90d>=500); score=stale*visible*log1p(impressions_90d)',
    'reason_codes': ['stale_visible_page', 'general_review'],
    'action_labels': ['refresh', 'monitor'],
    'leakage_note': 'trend_pct, trend_direction, is_declining_label and future-window fields were not used in score construction.'
}
(OUT_DIR / 'w04_baseline_receipt.json').write_text(json.dumps(receipt, indent=2))

display(queue.head(10))
print(f'Wrote {len(queue):,} rows to {OUT_DIR / "baseline_action_score.csv"}')

Base rate: 0.542
Precision@10: 0.800
Precision@50: 0.680


,rank,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days,word_count
0,1,content_cf56e2e2e282,client_7f2253d7e2,11.029699,stale_visible_page,refresh,61678,194,19.7,0.15,231,5125.0
1,2,content_7368877ea310,client_7f2253d7e2,10.993278,stale_visible_page,refresh,59472,194,24.8,0.13,231,2591.0
2,3,content_1bfaa38ff26c,client_7f2253d7e2,10.154869,stale_visible_page,refresh,25715,194,22.2,0.23,231,3861.0
3,4,content_0a91db491d14,client_7f2253d7e2,9.495519,stale_visible_page,refresh,13299,193,10.5,0.49,231,3478.0
4,5,content_5feee3994adb,client_7f2253d7e2,8.963544,stale_visible_page,refresh,7812,194,39.0,0.01,231,3590.0
5,6,content_c2d929d83eaa,client_7f2253d7e2,8.930494,stale_visible_page,refresh,7558,193,17.9,0.20,231,4758.0
6,7,content_b16bd7307b39,client_7f2253d7e2,8.431853,stale_visible_page,refresh,4590,194,31.0,0.00,231,4329.0
7,8,content_fe16a55cd13d,client_7f2253d7e2,8.424420,stale_visible_page,refresh,4556,194,16.4,0.33,231,3388.0
8,9,content_ecb6215e79fd,client_7f2253d7e2,8.396155,stale_visible_page,refresh,4429,194,25.3,0.38,231,4486.0
9,10,content_928af3e22c80,client_7f2253d7e2,7.437206,stale_visible_page,refresh,1697,193,15.8,0.12,231,3118.0


Wrote 30,000 rows to /Users/hardikk/Desktop/ML_Pipeline/ML_Pipeline/work/outputs/baseline_action_score.csv


## 3. Top-20 skeptic review

The assignment asks for the top ten; the skeleton asks for top twenty. I review twenty because it makes weak picks easier to spot. Each line names the action, why it ranked, and a concrete condition that would make the recommendation wrong.

In [4]:
top20 = work.head(20).copy()
review_rows = []
for _, r in top20.iterrows():
    if r['reason_code'] == 'stale_visible_page':
        why = f"old update ({int(r['days_since_last_update'])}d) + visible ({int(r['impressions_90d']):,} impressions)"
        wrong = 'wrong if the page is intentionally evergreen, already scheduled for refresh, or the traffic is low-quality/unconvertible.'
        action = 'refresh'
    else:
        why = 'does not meet both baseline gates; kept only as a low-priority comparator'
        wrong = 'wrong if manual review reveals a hidden business priority that the observable search metrics miss.'
        action = 'monitor'
    review_rows.append({
        'rank': int(r['rank']),
        'action': action,
        'why_it_is_here': why,
        'what_would_make_it_wrong': wrong
    })
review = pd.DataFrame(review_rows)
display(review)

,rank,action,why_it_is_here,what_would_make_it_wrong
0,1,refresh,"old update (194d) + visible (61,678 impressions)","wrong if the page is intentionally evergreen, ..."
1,2,refresh,"old update (194d) + visible (59,472 impressions)","wrong if the page is intentionally evergreen, ..."
2,3,refresh,"old update (194d) + visible (25,715 impressions)","wrong if the page is intentionally evergreen, ..."
3,4,refresh,"old update (193d) + visible (13,299 impressions)","wrong if the page is intentionally evergreen, ..."
4,5,refresh,"old update (194d) + visible (7,812 impressions)","wrong if the page is intentionally evergreen, ..."
5,6,refresh,"old update (193d) + visible (7,558 impressions)","wrong if the page is intentionally evergreen, ..."
6,7,refresh,"old update (194d) + visible (4,590 impressions)","wrong if the page is intentionally evergreen, ..."
7,8,refresh,"old update (194d) + visible (4,556 impressions)","wrong if the page is intentionally evergreen, ..."
8,9,refresh,"old update (194d) + visible (4,429 impressions)","wrong if the page is intentionally evergreen, ..."
9,10,refresh,"old update (193d) + visible (1,697 impressions)","wrong if the page is intentionally evergreen, ..."


## 4. Weak picks + leakage check

The most suspicious baseline picks are pages with high impressions but no compelling editorial opportunity beyond staleness. The rule is therefore a review queue, not an automatic publish/replace command.

**Leakage check:** the score uses only `days_since_last_update` and `impressions_90d`. It does not use `trend_pct`, `trend_direction`, `is_declining_label`, model outputs, product flags, URLs, titles, keywords, or any forward window. `avg_position` and `ctr` are retained in the exported queue only for human review; they are not score inputs.

In [5]:
score_inputs = ['days_since_last_update','impressions_90d']
forbidden_inputs = ['trend_pct','trend_direction','is_declining_label','future','next','label']
print('Score inputs:', score_inputs)
print('Forbidden/label-derived inputs are not referenced in the score expression.')
print('Weak-pick note: inspect the lowest-confidence rows manually before acting; stale + visible can still be the wrong editorial choice.')
print(f'Top-10 review rows available: {min(10, len(work))}')
print('Queue file exists:', (OUT_DIR / 'baseline_action_score.csv').exists())
assert not any(x in "days_since_last_update impressions_90d" for x in forbidden_inputs)

Score inputs: ['days_since_last_update', 'impressions_90d']
Forbidden/label-derived inputs are not referenced in the score expression.
Weak-pick note: inspect the lowest-confidence rows manually before acting; stale + visible can still be the wrong editorial choice.
Top-10 review rows available: 10
Queue file exists: True


## Self-check

- [x] Two visible bucket tables with `n`; at least one signal is flag-linked (staleness / refresh logic).
- [x] One transparent rule with a score, exactly one reason code per row, and an action label.
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`.
- [x] Top-20 skeptic review includes action, why, and what would make it wrong.
- [x] No label-derived or future-window fields are used as score inputs.
- [x] CSV is left uncommitted by design; the JSON receipt is a small reproducible run artifact.

**Important:** run this notebook top-to-bottom in Colab or locally, save the executed `.ipynb`, then commit it to the repo.